# 03 — Data Quality Assessment

**Project:** Retail Intelligence — Customer, Sales & Demand Analytics
**Phase:** 1 — Data Understanding (Task 3)
**Input:** the 9 raw Olist CSV files in `data/raw/`
**Builds on:** `01_dataset_inventory.ipynb` (structure) and `02_relationship_analysis.ipynb` (keys and relationships)

## Objective

A systematic, evidence-based assessment of data quality across all nine datasets, covering:

missing values · duplicates · data types · dates · numerical validity · categorical consistency · outliers · referential integrity

The output is a **data quality rule register** (Section 10). For each issue it records how many records are affected, how severe it is, and a recommended treatment for the cleaning phase.

## This is an assessment, not a cleaning step

| Rule | How it is enforced |
|---|---|
| Original files are never modified | Read-only; SHA-256 fingerprints are compared at the end |
| Loaded data is never modified | Every check works on **temporary derived Series** (e.g. parsed dates, flags). A hash of each in-memory table is taken after loading and re-checked at the end |
| No deletion, filling, outlier removal or saved conversions | None of `drop_duplicates`, `dropna`, `fillna` or `to_csv`/`to_parquet` is used on any table |
| No processed datasets | A snapshot of `data/processed/` is compared at the end |
| Unusual ≠ wrong | Every unusual value is classified as **invalid** (impossible), **suspicious** (implausible, needs review) or **plausible extreme** (real but rare) before any treatment is recommended |

## Severity scale used in the rule register

| Severity | Meaning |
|---|---|
| **High** | If untreated, core metrics (revenue, customer counts, delivery performance) will be **wrong**, or analysis is blocked |
| **Medium** | Affects a specific analysis or a noticeable subset of records; needs an explicit decision |
| **Low** | Cosmetic, or affects very few records; minor effect on results |
| **Info** | Not an error. A structural property of the data that analysts must know about |

## 1. Setup and loading

### 1.1 Environment and paths

In [1]:
import hashlib
import re
import sys
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(f"No project root containing data/raw/ above {start}")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Python : {sys.version.split()[0]}  ({sys.executable})")
print(f"pandas : {pd.__version__} | numpy : {np.__version__} | scipy : {__import__('scipy').__version__}")
print(f"In .venv: {'.venv' in Path(sys.prefix).parts} | Project root: {PROJECT_ROOT}")

Python : 3.12.10  (C:\Users\Admin\Desktop\retail-intelligence\.venv\Scripts\python.exe)
pandas : 3.1.0 | numpy : 2.5.4 | scipy : 1.18.1
In .venv: True | Project root: C:\Users\Admin\Desktop\retail-intelligence


### 1.2 Fingerprints before loading

SHA-256 hashes of the raw files and a listing of `data/processed/` are taken **before** any data is read.

In [2]:
FILES = {
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

sha_before = {f: sha256(RAW_DIR / f) for f in FILES.values()}
processed_before = sorted(p.name for p in PROCESSED_DIR.rglob("*")) if PROCESSED_DIR.exists() else []
display(pd.Series(sha_before, name="sha256").to_frame())
print("data/processed/ contents before:", processed_before or "empty")

,sha256
olist_customers_dataset.csv,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2
olist_geolocation_dataset.csv,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6
olist_orders_dataset.csv,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf
olist_order_items_dataset.csv,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279
olist_order_payments_dataset.csv,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0
olist_order_reviews_dataset.csv,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7
olist_products_dataset.csv,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc
olist_sellers_dataset.csv,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e
product_category_name_translation.csv,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278


data/processed/ contents before: empty


### 1.3 Load all nine datasets (once)

**Decision: load with pandas' default type inference**, as in notebooks 01–02. Section 4 assesses exactly what this inference gets wrong (dates read as text, zip codes read as numbers). Assessing those problems requires seeing the data as a naive load would deliver it.

Right after loading, a content hash of each in-memory DataFrame is stored. Section 11 recomputes it to prove that no check changed the loaded data.

In [3]:
datasets: dict[str, pd.DataFrame] = {}
load_errors: dict[str, str] = {}
for alias, fname in FILES.items():
    try:
        datasets[alias] = pd.read_csv(RAW_DIR / fname)
    except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeDecodeError) as exc:
        load_errors[alias] = f"{type(exc).__name__}: {exc}"
assert not load_errors, load_errors

def frame_hash(df: pd.DataFrame) -> str:
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()

memory_hash_before = {a: frame_hash(d) for a, d in datasets.items()}

# Short aliases used throughout (references, not copies — they are never modified)
customers, geo, orders, items = (datasets[k] for k in ["customers", "geolocation", "orders", "order_items"])
payments, reviews, products, sellers, translation = (datasets[k] for k in ["payments", "reviews", "products", "sellers", "translation"])

display(pd.DataFrame({a: {"rows": len(d), "columns": d.shape[1]} for a, d in datasets.items()}).T)

,rows,columns
customers,99441,5
geolocation,1000163,5
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
products,32951,9
sellers,3095,4
translation,71,2


### 1.4 Data-quality rule register

Every issue found below is recorded with `register(...)` **in the cell where it is measured**. The affected-record count in the final register therefore always comes from the code, never from hand-typed numbers.

In [4]:
RULES: list[dict] = []
SEVERITY_ORDER = {"High": 0, "Medium": 1, "Low": 2, "Info": 3}

def register(dataset, rule, issue, affected, severity, treatment, justification, section):
    assert severity in SEVERITY_ORDER, severity
    RULES.append({"dataset": dataset, "column_or_rule": rule, "quality_issue": issue,
                  "affected_records": int(affected), "pct_of_dataset": None, "severity": severity,
                  "recommended_treatment": treatment, "justification": justification, "section": section})

## 2. Missing values

### 2.1 Counts and percentages for every column

Only the columns that have missing values are shown. All other columns in all nine datasets are complete.

In [5]:
missing_rows = []
for alias, df in datasets.items():
    n = len(df)
    for col in df.columns:
        k = int(df[col].isna().sum())
        missing_rows.append({"dataset": alias, "column": col, "rows": n, "missing": k, "missing_pct": round(k / n * 100, 3)})
missing_df = pd.DataFrame(missing_rows)

display(missing_df[missing_df["missing"] > 0].sort_values("missing_pct", ascending=False).reset_index(drop=True))
print("Datasets with no missing values at all:",
      sorted(set(missing_df["dataset"]) - set(missing_df.loc[missing_df["missing"] > 0, "dataset"])))

,dataset,column,rows,missing,missing_pct
0,reviews,review_comment_title,99224,87656,88.342
1,reviews,review_comment_message,99224,58247,58.703
2,orders,order_delivered_customer_date,99441,2965,2.982
3,products,product_name_lenght,32951,610,1.851
4,products,product_category_name,32951,610,1.851
5,products,product_description_lenght,32951,610,1.851
6,products,product_photos_qty,32951,610,1.851
7,orders,order_delivered_carrier_date,99441,1783,1.793
8,orders,order_approved_at,99441,160,0.161
9,products,product_weight_g,32951,2,0.006


Datasets with no missing values at all: ['customers', 'geolocation', 'order_items', 'payments', 'sellers', 'translation']


### 2.2 Distribution of missing values: do gaps occur together?

A column-by-column count does not show *structure*. If gaps occur together in the same rows, they share one cause, so the **co-occurrence pattern** of missing values is examined for each affected table.

In [6]:
def missing_patterns(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df.columns if df[c].isna().any()]
    pat = df[cols].isna().value_counts().rename("rows").reset_index()
    return pat.replace({True: "MISSING", False: "·"})

for alias in ["products", "orders", "reviews"]:
    print(f"--- {alias}")
    display(missing_patterns(datasets[alias]))

--- products


,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,rows
0,·,·,·,·,·,·,·,·,32340
1,MISSING,MISSING,MISSING,MISSING,·,·,·,·,609
2,·,·,·,·,MISSING,MISSING,MISSING,MISSING,1
3,MISSING,MISSING,MISSING,MISSING,MISSING,MISSING,MISSING,MISSING,1


--- orders


,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,rows
0,·,·,·,96461
1,·,MISSING,MISSING,1636
2,·,·,MISSING,1183
3,MISSING,MISSING,MISSING,146
4,MISSING,·,·,14
5,·,MISSING,·,1


--- reviews


,review_comment_title,review_comment_message,rows
0,MISSING,MISSING,56518
1,MISSING,·,31138
2,·,·,9839
3,·,MISSING,1729


### 2.3 Potential business meaning of the gaps

Missing values are interpreted **against other columns** rather than assumed to be errors.

- **Orders:** a missing timestamp may simply mean the event has not happened yet (an order that has not shipped has no carrier date). Gaps are therefore cross-tabulated against `order_status`.
- **Reviews:** comments may be optional. The rate of missing comments is compared across review scores.

In [7]:
order_ts_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
status_missing = (orders[order_ts_cols].isna()
                  .groupby(orders["order_status"]).sum()
                  .join(orders["order_status"].value_counts().rename("orders")))
status_missing = status_missing[["orders", *order_ts_cols]]
display(status_missing.sort_values("orders", ascending=False))

,orders,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [8]:
review_missing_by_score = (reviews[["review_comment_title", "review_comment_message"]].isna()
                           .groupby(reviews["review_score"]).mean().mul(100).round(1)
                           .rename(columns=lambda c: f"{c}_missing_%"))
display(review_missing_by_score)
r, p = stats.spearmanr(reviews["review_score"], reviews["review_comment_message"].isna())
print(f"Spearman correlation (score vs. message missing): rho={r:.3f}, p={p:.1e}")

,review_comment_title_missing_%,review_comment_message_missing_%
review_score,,
1,83.6,23.5
2,84.8,31.9
3,89.9,56.5
4,90.9,68.8
5,88.4,64.1


Spearman correlation (score vs. message missing): rho=0.196, p=0.0e+00


In [9]:
# Products: is missing category concentrated in sales, or in rarely sold products?
sold_per_product = items["product_id"].value_counts()
prod_missing_cat = products["product_category_name"].isna()
cmp = pd.DataFrame({
    "products": [prod_missing_cat.sum(), (~prod_missing_cat).sum()],
    "order_items": [items["product_id"].isin(products.loc[prod_missing_cat, "product_id"]).sum(),
                    items["product_id"].isin(products.loc[~prod_missing_cat, "product_id"]).sum()],
}, index=["category missing", "category present"])
cmp["items_per_product"] = (cmp["order_items"] / cmp["products"]).round(2)
display(cmp)

,products,order_items,items_per_product
category missing,610,1603,2.63
category present,32341,111047,3.43


In [10]:
# Register missing-value findings
delivered = orders["order_status"].eq("delivered")
register("orders", "order_approved_at", "Missing approval timestamp", orders["order_approved_at"].isna().sum(), "Low",
         "Keep as missing; for 'delivered' orders consider flagging rather than imputing",
         "Mostly canceled/created orders (event never happened); 14 are delivered orders, a genuine gap", "2")
register("orders", "order_delivered_carrier_date / _customer_date", "Missing logistics timestamps (status-consistent)",
         orders["order_delivered_customer_date"].isna().sum(), "Info",
         "Keep as missing; restrict delivery-time metrics to delivered orders with both dates",
         "Gaps follow order_status (shipped, invoiced, processing, canceled...) — events not yet occurred", "2")
register("reviews", "review_comment_title / review_comment_message", "Optional free-text not provided",
         reviews["review_comment_message"].isna().sum(), "Info",
         "Keep as missing; treat 'has comment' as a property, never impute text",
         "Comments are optional; missingness rises with review score (satisfied customers comment less)", "2")
register("products", "category, name/description length, photos", "Product metadata missing (same rows)",
         prod_missing_cat.sum(), "Medium",
         "Keep rows; label category as an explicit 'unknown' group in reporting",
         "4 columns missing together = one cause (unpopulated listing); products are still sold and must not be dropped", "2")
register("products", "weight and 3 dimensions", "Physical attributes missing", products["product_weight_g"].isna().sum(), "Low",
         "Keep as missing; exclude from logistics/freight modelling only", "Only 2 products affected", "2")

## 3. Duplicates

Four kinds of duplication are distinguished, because each has a different cause and treatment:

| Type | Definition |
|---|---|
| **Exact duplicate rows** | Rows identical in every column |
| **Duplicate identifiers** | Repeated values in a column that is supposed to identify a row |
| **Business-level duplicates** | Rows identical **apart from their identifier / sequence column**: possibly the same real-world event recorded twice, or a legitimate repetition |
| **Composite key violations** | Repeated combinations of the composite keys established in notebook 02 |

### 3.1 Exact duplicate rows

In [11]:
exact_dup = pd.DataFrame({alias: {"rows": len(df), "exact_duplicate_rows": int(df.duplicated().sum()),
                                  "pct": round(df.duplicated().mean() * 100, 3)}
                          for alias, df in datasets.items()}).T
display(exact_dup)
register("geolocation", "all columns", "Exact duplicate rows", exact_dup.loc["geolocation", "exact_duplicate_rows"], "Medium",
         "Do not use row-level; aggregate to one row per zip prefix (e.g. median lat/lng) when building a lookup",
         "26% repeated points add no information but would multiply joins", "3.1")

,rows,exact_duplicate_rows,pct
customers,99441.0,0.0,0.000
geolocation,1000163.0,261831.0,26.179
orders,99441.0,0.0,0.000
order_items,112650.0,0.0,0.000
payments,103886.0,0.0,0.000
reviews,99224.0,0.0,0.000
products,32951.0,0.0,0.000
sellers,3095.0,0.0,0.000
translation,71.0,0.0,0.000


### 3.2 Duplicate identifiers

Identifier columns are tested where they should identify a row. Columns that legitimately repeat (foreign keys such as `order_items.order_id`) are excluded, because notebook 02 established them as one-to-many.

In [12]:
ID_COLUMNS = [("customers", "customer_id"), ("customers", "customer_unique_id"), ("orders", "order_id"),
              ("products", "product_id"), ("sellers", "seller_id"), ("reviews", "review_id"),
              ("translation", "product_category_name"), ("geolocation", "geolocation_zip_code_prefix")]
dup_ids = pd.DataFrame([{
    "dataset": t, "column": c, "rows": len(datasets[t]), "distinct": datasets[t][c].nunique(),
    "values_repeated": int((datasets[t][c].value_counts() > 1).sum()),
    "rows_with_repeated_value": int(datasets[t][c].duplicated(keep=False).sum()),
} for t, c in ID_COLUMNS])
display(dup_ids)

rev_rep = dup_ids.query("dataset == 'reviews'").iloc[0]
register("reviews", "review_id", "review_id not unique (one review linked to several orders)", rev_rep["rows_with_repeated_value"],
         "Medium", "Key reviews on (review_id, order_id); de-duplicate by review_id only for review-level counts",
         "Content identical across repeats (nb 02) — counting rows would over-count reviews", "3.2")
cu_rep = customers["customer_unique_id"].duplicated(keep=False).sum()
register("customers", "customer_unique_id", "One person has several customer_id (one per order)", cu_rep, "High",
         "Use customer_unique_id as the customer key for all customer-level metrics",
         "customer_id is order-level (nb 02); using it makes every customer look like a one-time buyer", "3.2")
register("geolocation", "geolocation_zip_code_prefix", "Zip prefix repeats (many points per zip)",
         geo["geolocation_zip_code_prefix"].duplicated(keep=False).sum(), "Info",
         "Treat geolocation as point-level; never join on zip without aggregating first",
         "Not an identifier — structural property of the table", "3.2")

,dataset,column,rows,distinct,values_repeated,rows_with_repeated_value
0,customers,customer_id,99441,99441,0,0
1,customers,customer_unique_id,99441,96096,2997,6342
2,orders,order_id,99441,99441,0,0
3,products,product_id,32951,32951,0,0
4,sellers,seller_id,3095,3095,0,0
5,reviews,review_id,99224,98410,789,1603
6,translation,product_category_name,71,71,0,0
7,geolocation,geolocation_zip_code_prefix,1000163,19015,17972,999120


### 3.3 Business-level duplicates

For each table, the rows are compared **ignoring the identifier or sequence column**. Each result is then interpreted: some repetitions are *expected* (one row per unit purchased), others are *suspicious* (the same person ordering twice in the same second).

In [13]:
BUSINESS_DUP_TESTS = [  # (dataset, columns that define 'the same event', interpretation hint)
    ("order_items", ["order_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"],
     "same product/price repeated in one order — quantity is stored as rows"),
    ("payments", ["order_id", "payment_type", "payment_installments", "payment_value"],
     "identical payment repeated within an order (e.g. several equal vouchers)"),
    ("reviews", ["order_id", "review_score", "review_comment_title", "review_comment_message", "review_creation_date"],
     "same review content on one order under different review_ids"),
    ("customers", ["customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
     "same person at same address under another customer_id (expected: id is per order)"),
    ("products", [c for c in products.columns if c != "product_id"],
     "different product_ids with identical attributes"),
    ("sellers", ["seller_zip_code_prefix", "seller_city", "seller_state"],
     "different sellers at the same zip/city — plausible (shared business areas)"),
]
biz_rows = []
for t, cols, hint in BUSINESS_DUP_TESTS:
    d = datasets[t].duplicated(subset=cols)
    biz_rows.append({"dataset": t, "compared_on": ", ".join(cols), "business_duplicates": int(d.sum()),
                     "pct": round(d.mean() * 100, 2), "interpretation": hint})
biz_df = pd.DataFrame(biz_rows)
display(biz_df[["dataset", "business_duplicates", "pct", "interpretation"]])

,dataset,business_duplicates,pct,interpretation
0,order_items,10225,9.08,same product/price repeated in one order — quantity is stored as rows
1,payments,614,0.59,identical payment repeated within an order (e.g. several equal vouchers)
2,reviews,86,0.09,same review content on one order under different review_ids
3,customers,3089,3.11,same person at same address under another customer_id (expected: id is per order)
4,products,695,2.11,different product_ids with identical attributes
5,sellers,799,25.82,different sellers at the same zip/city — plausible (shared business areas)


In [14]:
# Products: are 'identical' products mostly rows with missing attributes (where NaN==NaN makes them look identical)?
prod_cols = [c for c in products.columns if c != "product_id"]
prod_dup = products.duplicated(subset=prod_cols, keep=False)
print(f"products in identical-attribute groups: {prod_dup.sum():,}  "
      f"of which category missing: {(prod_dup & products['product_category_name'].isna()).sum():,}")

# Orders: same person, same purchase second — split orders or double submissions?
oc = orders[["order_id", "customer_id", "order_purchase_timestamp", "order_status"]].merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left", validate="one_to_one")
same_second = oc.duplicated(subset=["customer_unique_id", "order_purchase_timestamp"], keep=False)
grp = oc[same_second].groupby(["customer_unique_id", "order_purchase_timestamp"])
seller_counts = (items[items["order_id"].isin(oc.loc[same_second, "order_id"])]
                 .merge(oc[["order_id", "customer_unique_id", "order_purchase_timestamp"]], on="order_id")
                 .groupby(["customer_unique_id", "order_purchase_timestamp"])["seller_id"].nunique())
print(f"orders placed by the same person in the same second: {same_second.sum():,} rows in {grp.ngroups:,} groups")
print(f"... groups whose orders involve >1 distinct seller : {(seller_counts > 1).mean():.1%}")

register("orders", "customer_unique_id + order_purchase_timestamp", "Same person, several orders in the same second",
         same_second.sum(), "Low",
         "Keep as separate orders; optionally group into a 'basket' for basket analysis; review single-seller groups",
         "59% of groups span several sellers (consistent with a checkout split per seller); the other 41% are unexplained — possible double submissions", "3.3")
register("reviews", "order_id + score + title + message + creation date", "Same review content on one order under different review_ids",
         biz_df.query("dataset=='reviews'")["business_duplicates"].iloc[0], "Low",
         "Flag; count one review per (order_id, content) in satisfaction metrics",
         "Identical content and date on the same order — likely duplicate submissions", "3.3")
register("products", "all attributes except product_id", "Different product_ids with identical attributes",
         biz_df.query("dataset=='products'")["business_duplicates"].iloc[0], "Info",
         "Keep; do not merge products (attributes are coarse, so collisions are expected)",
         "Same category, text lengths, photos and dimensions can legitimately coincide (e.g. variants of one item)", "3.3")
register("order_items", "order_id + product_id + seller_id + price", "Repeated identical item rows within an order",
         biz_df.query("dataset=='order_items'")["business_duplicates"].iloc[0], "Info",
         "Keep; derive quantity as row count per (order_id, product_id)",
         "Quantity is represented by repeated rows (no quantity column)", "3.3")

products in identical-attribute groups: 1,158  of which category missing: 133


orders placed by the same person in the same second: 582 rows in 290 groups
... groups whose orders involve >1 distinct seller : 59.0%


### 3.4 Composite key violations

In [15]:
COMPOSITE_KEYS = {"order_items": ["order_id", "order_item_id"], "payments": ["order_id", "payment_sequential"],
                  "reviews": ["review_id", "order_id"]}
ck = pd.DataFrame({t: {"key": " + ".join(k), "violations": int(datasets[t].duplicated(subset=k).sum()),
                       "nulls_in_key": int(datasets[t][k].isna().any(axis=1).sum())}
                   for t, k in COMPOSITE_KEYS.items()}).T
display(ck)
assert (ck["violations"] == 0).all() and (ck["nulls_in_key"] == 0).all()
print("No composite-key violations.")

,key,violations,nulls_in_key
order_items,order_id + order_item_id,0,0
payments,order_id + payment_sequential,0,0
reviews,review_id + order_id,0,0


No composite-key violations.


## 4. Data types

Each column's **inferred** type is compared with its **semantic** type, meaning what the values represent. The semantic types are assigned from the column contents (examined in notebook 01), and each mismatch is then **tested** rather than assumed.

In [16]:
SEMANTIC = {}
for alias, df in datasets.items():
    for col in df.columns:
        if col.endswith("_id") and col != "order_item_id":
            SEMANTIC[(alias, col)] = "identifier (text)"
        elif "zip_code_prefix" in col:
            SEMANTIC[(alias, col)] = "postal code (text)"
        elif any(k in col for k in ["_date", "_timestamp", "_at"]):
            SEMANTIC[(alias, col)] = "datetime"
        elif col in ["order_item_id", "payment_sequential", "payment_installments", "review_score",
                     "product_name_lenght", "product_description_lenght", "product_photos_qty"]:
            SEMANTIC[(alias, col)] = "integer count / ordinal"
        elif df[col].dtype.kind == "f":
            SEMANTIC[(alias, col)] = "continuous number"
        else:
            SEMANTIC[(alias, col)] = "category / text"

def type_status(alias, col, sem):
    s = datasets[alias][col]
    k = s.dtype.kind
    if sem == "datetime":
        return "MISMATCH: stored as text"
    if sem == "postal code (text)":
        return "MISMATCH: stored as number" if k in "iuf" else "ok"
    if sem == "integer count / ordinal":
        return "ok" if k in "iu" else "MISMATCH: float (because of missing values)"
    return "ok"

types_df = pd.DataFrame([{"dataset": a, "column": c, "inferred_dtype": str(datasets[a][c].dtype),
                          "semantic_type": s, "status": type_status(a, c, s)} for (a, c), s in SEMANTIC.items()])
display(types_df[types_df["status"] != "ok"].reset_index(drop=True))
print(f"columns checked: {len(types_df)} | mismatches: {(types_df['status'] != 'ok').sum()}")

,dataset,column,inferred_dtype,semantic_type,status
0,customers,customer_zip_code_prefix,int64,postal code (text),MISMATCH: stored as number
1,geolocation,geolocation_zip_code_prefix,int64,postal code (text),MISMATCH: stored as number
2,orders,order_purchase_timestamp,str,datetime,MISMATCH: stored as text
3,orders,order_approved_at,str,datetime,MISMATCH: stored as text
4,orders,order_delivered_carrier_date,str,datetime,MISMATCH: stored as text
5,orders,order_delivered_customer_date,str,datetime,MISMATCH: stored as text
6,orders,order_estimated_delivery_date,str,datetime,MISMATCH: stored as text
7,order_items,shipping_limit_date,str,datetime,MISMATCH: stored as text
8,reviews,review_creation_date,str,datetime,MISMATCH: stored as text
9,reviews,review_answer_timestamp,str,datetime,MISMATCH: stored as text


columns checked: 52 | mismatches: 14


### 4.1 Date and timestamp formats

All date columns are text. Before anything else, the **format** must be confirmed to be uniform. A strict regular expression is applied to every non-missing value, then each value is parsed with an **explicit format** (`errors="coerce"`). Any value that is not a real calendar date (e.g. 2018-02-30) becomes an unparseable count instead of being silently guessed.

**Decision:** the parsed timestamps are kept in a separate dictionary `ts` for analysis. They are never written back to the DataFrames.

In [17]:
DATE_COLS = [(a, c) for (a, c), s in SEMANTIC.items() if s == "datetime"]
FMT = "%Y-%m-%d %H:%M:%S"
ts: dict[tuple[str, str], pd.Series] = {}
fmt_rows = []
for a, c in DATE_COLS:
    raw = datasets[a][c]
    present = raw.notna()
    pattern_ok = raw[present].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}")
    parsed = pd.to_datetime(raw, format=FMT, errors="coerce")
    ts[(a, c)] = parsed
    fmt_rows.append({"dataset": a, "column": c, "non_missing": int(present.sum()),
                     "match_format": int(pattern_ok.sum()), "unparseable": int((present & parsed.isna()).sum()),
                     "min": parsed.min(), "max": parsed.max(),
                     "time_always_00:00:00": bool((parsed.dropna().dt.normalize() == parsed.dropna()).all())})
fmt_df = pd.DataFrame(fmt_rows)
display(fmt_df)

register("orders / order_items / reviews", "8 date/time columns", "Timestamps stored as text", len(orders) + len(items) + len(reviews),
         "High", "Parse with explicit format '%Y-%m-%d %H:%M:%S' (errors='coerce') at load time",
         "All values match one format and parse cleanly, but as text they cannot be compared, sorted or differenced", "4.1")

,dataset,column,non_missing,match_format,unparseable,min,max,time_always_00:00:00
0,orders,order_purchase_timestamp,99441,99441,0,2016-09-04 21:15:19,2018-10-17 17:30:18,False
1,orders,order_approved_at,99281,99281,0,2016-09-15 12:16:38,2018-09-03 17:40:06,False
2,orders,order_delivered_carrier_date,97658,97658,0,2016-10-08 10:34:01,2018-09-11 19:48:28,False
3,orders,order_delivered_customer_date,96476,96476,0,2016-10-11 13:46:32,2018-10-17 13:22:46,False
4,orders,order_estimated_delivery_date,99441,99441,0,2016-09-30 00:00:00,2018-11-12 00:00:00,True
5,order_items,shipping_limit_date,112650,112650,0,2016-09-19 00:15:34,2020-04-09 22:35:08,False
6,reviews,review_creation_date,99224,99224,0,2016-10-02 00:00:00,2018-08-31 00:00:00,False
7,reviews,review_answer_timestamp,99224,99224,0,2016-10-07 18:32:28,2018-10-29 12:27:35,False


`order_estimated_delivery_date` always has time **00:00:00**, and `review_creation_date` almost always does (checked in the next cell). These are effectively **date-only** values stored in timestamp form. Comparing them with true timestamps on the same day would wrongly flag same-day events, so Section 5 compares them at day level.

In [18]:
for key in [("orders", "order_estimated_delivery_date"), ("reviews", "review_creation_date")]:
    s = ts[key].dropna()
    print(f"{key[0]}.{key[1]}: values with a non-midnight time = {(s != s.dt.normalize()).sum():,} of {len(s):,}")

orders.order_estimated_delivery_date: values with a non-midnight time = 0 of 99,441
reviews.review_creation_date: values with a non-midnight time = 85 of 99,224


### 4.2 Numeric values stored as text

Every text column, except identifiers and dates, is tested: what share of its non-missing values would parse as a number?

In [19]:
num_as_text = []
for alias, df in datasets.items():
    for col in df.columns:
        if (pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col])) \
                and SEMANTIC[(alias, col)] not in ("identifier (text)", "datetime"):
            s = df[col].dropna()
            numeric_share = pd.to_numeric(s, errors="coerce").notna().mean()
            num_as_text.append({"dataset": alias, "column": col, "numeric_share_%": round(numeric_share * 100, 3),
                                "numeric_values": int(round(numeric_share * len(s)))})
num_as_text_df = pd.DataFrame(num_as_text)
display(num_as_text_df[num_as_text_df["numeric_values"] > 0])
print("Text columns that are predominantly numeric (>95%):",
      num_as_text_df.loc[num_as_text_df["numeric_share_%"] > 95, "column"].tolist() or "none")

,dataset,column,numeric_share_%,numeric_values
6,reviews,review_comment_title,2.732,316
7,reviews,review_comment_message,0.071,29
9,sellers,seller_city,0.032,1


Text columns that are predominantly numeric (>95%): none


No column is "numeric stored as text". The only numeric-looking text values are a few **city names that are numbers**, which is a categorical problem (Section 6). The reverse problem (text codes read as numbers) is the important one, and is examined next.

### 4.3 Identifiers interpreted as numbers: zip-code prefixes

**Decision:** to find out whether the raw file stores zip prefixes *with* leading zeros, the three zip columns are re-read **as text** (`dtype=str`, only that column). This is the only re-read in the notebook, and it is required: the default load has already discarded the information being tested.

In [20]:
ZIP_COLS = [("customers", "customer_zip_code_prefix"), ("sellers", "seller_zip_code_prefix"),
            ("geolocation", "geolocation_zip_code_prefix")]
zip_rows = []
for a, c in ZIP_COLS:
    as_text = pd.read_csv(RAW_DIR / FILES[a], usecols=[c], dtype=str)[c]
    as_int = datasets[a][c]
    zip_rows.append({"dataset": a, "column": c, "inferred_dtype": str(as_int.dtype),
                     "raw_text_lengths": as_text.str.len().value_counts().sort_index().to_dict(),
                     "raw_with_leading_zero": int(as_text.str.startswith("0").sum()),
                     "int_values_below_10000": int((as_int < 10000).sum()),
                     "text_equals_int_zfill5": bool((as_int.astype(str).str.zfill(5) == as_text).all())})
zip_df = pd.DataFrame(zip_rows)
display(zip_df)
register("customers / sellers / geolocation", "*_zip_code_prefix", "Zip prefix inferred as int64 — leading zeros lost",
         zip_df["int_values_below_10000"].sum(), "High",
         "Read zip columns with dtype=str at load time (on every table, so joins stay type-consistent)",
         "Codes like '01037' become 1037: wrong as labels, and mismatched if any other source keeps them as text", "4.3")

,dataset,column,inferred_dtype,raw_text_lengths,raw_with_leading_zero,int_values_below_10000,text_equals_int_zfill5
0,customers,customer_zip_code_prefix,int64,{5: 99441},23995,23995,True
1,sellers,seller_zip_code_prefix,int64,{5: 3095},1027,1027,True
2,geolocation,geolocation_zip_code_prefix,int64,{5: 1000163},245733,245733,True


### 4.4 Integer counts stored as float, and source column names

In [21]:
float_ints = []
for col in ["product_name_lenght", "product_description_lenght", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    s = products[col].dropna()
    float_ints.append({"column": col, "dtype": str(products[col].dtype), "missing": int(products[col].isna().sum()),
                       "non_integer_values": int((s % 1 != 0).sum())})
display(pd.DataFrame(float_ints))
register("products", "7 numeric attribute columns", "Whole-number values stored as float64 (because of NaN)", len(products), "Low",
         "Cast to nullable integer (Int64) when cleaning", "No fractional values exist; float is an artefact of missing values", "4.4")
register("products", "product_name_lenght / product_description_lenght", "Misspelled source column names ('lenght')", len(products), "Low",
         "Rename to *_length in the processed layer; keep a mapping to the raw names", "Prevents downstream typos; no data change", "4.4")

,column,dtype,missing,non_integer_values
0,product_name_lenght,float64,610,0
1,product_description_lenght,float64,610,0
2,product_photos_qty,float64,610,0
3,product_weight_g,float64,2,0
4,product_length_cm,float64,2,0
5,product_height_cm,float64,2,0
6,product_width_cm,float64,2,0


## 5. Date validation

### 5.1 Invalid and out-of-range dates

Section 4.1 found **no unparseable dates**. Validity is therefore assessed against the **observed purchase window**: a date long after the last purchase, or before the first, is suspicious.

In [22]:
P = ts[("orders", "order_purchase_timestamp")]
A = ts[("orders", "order_approved_at")]
C = ts[("orders", "order_delivered_carrier_date")]
D = ts[("orders", "order_delivered_customer_date")]
E = ts[("orders", "order_estimated_delivery_date")]
SL = ts[("order_items", "shipping_limit_date")]
RC = ts[("reviews", "review_creation_date")]
RA = ts[("reviews", "review_answer_timestamp")]

window_start, window_end = P.min(), P.max()
print(f"Purchase window: {window_start} → {window_end}")
range_rows = []
for (a, c), s in ts.items():
    range_rows.append({"dataset": a, "column": c, "before_first_purchase": int((s < window_start).sum()),
                       "after_last_purchase": int((s > window_end).sum()),
                       "more_than_1y_after_last_purchase": int((s > window_end + pd.DateOffset(years=1)).sum())})
display(pd.DataFrame(range_rows))

Purchase window: 2016-09-04 21:15:19 → 2018-10-17 17:30:18


,dataset,column,before_first_purchase,after_last_purchase,more_than_1y_after_last_purchase
0,orders,order_purchase_timestamp,0,0,0
1,orders,order_approved_at,0,0,0
2,orders,order_delivered_carrier_date,0,0,0
3,orders,order_delivered_customer_date,0,0,0
4,orders,order_estimated_delivery_date,0,6,0
5,order_items,shipping_limit_date,0,4,4
6,reviews,review_creation_date,0,0,0
7,reviews,review_answer_timestamp,0,11,0


In [23]:
far_future = items.loc[SL > window_end + pd.DateOffset(years=1), ["order_id", "order_item_id", "shipping_limit_date", "price"]]
far_future = far_future.merge(orders[["order_id", "order_purchase_timestamp", "order_status"]], on="order_id")
display(far_future)
register("order_items", "shipping_limit_date", "Shipping deadline > 1 year after last purchase (year 2020)", len(far_future), "Low",
         "Flag as invalid; set to missing only in the processed layer if used",
         "Purchases are from 2016-2018; a 2020 deadline is not credible for these orders", "5.1")

,order_id,order_item_id,shipping_limit_date,price,order_purchase_timestamp,order_status
0,13bdf405f961a6deec817d817f5c6624,1,2020-02-05 03:30:51,69.99,2017-03-16 02:30:51,canceled
1,9c94a4ea2f7876660fa6f1b59b69c8e6,1,2020-02-03 20:23:22,75.99,2017-03-14 19:23:22,shipped
2,c2bb89b5c1dd978d507284be78a04cb2,1,2020-04-09 22:35:08,99.99,2017-05-23 22:28:36,delivered
3,c2bb89b5c1dd978d507284be78a04cb2,2,2020-04-09 22:35:08,99.99,2017-05-23 22:28:36,delivered


### 5.2 Chronological consistency of the order lifecycle

The expected sequence is **purchase → approval → handed to carrier → delivered to customer**. Each rule is evaluated only where **both** timestamps exist, so missing values are never counted as violations.

**Severity reasoning.**

- An event *before the purchase* is **impossible**.
- Delivery before carrier hand-off is **impossible**.
- Carrier hand-off before *approval* is **suspicious but explainable**: approval can be recorded late, for example for boleto payments that confirm after the parcel is prepared.

In [24]:
def rule(name, mask_violation, both_present, kind):
    return {"rule": name, "evaluated": int(both_present.sum()), "violations": int(mask_violation.sum()),
            "violation_%": round(mask_violation.sum() / max(both_present.sum(), 1) * 100, 3), "kind": kind}

chron = pd.DataFrame([
    rule("approved_at < purchase",                 A < P, A.notna() & P.notna(), "impossible"),
    rule("carrier_date < purchase",                C < P, C.notna() & P.notna(), "impossible"),
    rule("customer_date < purchase",               D < P, D.notna() & P.notna(), "impossible"),
    rule("customer_date < carrier_date",           D < C, D.notna() & C.notna(), "impossible"),
    rule("carrier_date < approved_at",             C < A, C.notna() & A.notna(), "suspicious"),
    rule("customer_date < approved_at",            D < A, D.notna() & A.notna(), "suspicious"),
    rule("estimated_delivery < purchase (day)",    E < P.dt.normalize(), E.notna(), "impossible"),
])
display(chron)

,rule,evaluated,violations,violation_%,kind
0,approved_at < purchase,99281,0,0.000,impossible
1,carrier_date < purchase,97658,166,0.170,impossible
2,customer_date < purchase,96476,0,0.000,impossible
3,customer_date < carrier_date,96475,23,0.024,impossible
4,carrier_date < approved_at,97644,1359,1.392,suspicious
5,customer_date < approved_at,96462,61,0.063,suspicious
6,estimated_delivery < purchase (day),99441,0,0.000,impossible


In [25]:
# How far back do the impossible carrier dates go? (size of the error informs treatment)
carrier_before_purchase = (C < P)
gap_h = ((P - C)[carrier_before_purchase].dt.total_seconds() / 3600)
print("carrier date before purchase — hours early:", gap_h.describe().round(1).to_dict())
deliv_before_carrier = (D < C)
gap_d = ((C - D)[deliv_before_carrier].dt.total_seconds() / 86400)
print("delivered before carrier — days early      :", gap_d.describe().round(1).to_dict())
cba = (C < A)
print("carrier before approval — hours early      :", ((A - C)[cba].dt.total_seconds() / 3600).describe().round(1).to_dict())

carrier date before purchase — hours early: {'count': 166.0, 'mean': 26.0, 'std': 318.9, 'min': 0.0, '25%': 0.2, '50%': 0.6, '75%': 1.0, 'max': 4109.1}
delivered before carrier — days early      : {'count': 23.0, 'mean': 3.3, 'std': 3.7, 'min': 0.0, '25%': 1.0, '50%': 1.7, '75%': 5.4, 'max': 16.1}
carrier before approval — hours early      : {'count': 1359.0, 'mean': 24.8, 'std': 115.3, 'min': 0.0, '25%': 1.4, '50%': 17.2, '75%': 26.0, 'max': 4109.3}


In [26]:
v = chron.set_index("rule")["violations"]
register("orders", "order_delivered_carrier_date < order_purchase_timestamp", "Carrier hand-off before purchase (impossible)",
         v["carrier_date < purchase"], "Medium", "Flag; exclude these orders from carrier-lead-time metrics",
         "Impossible sequence — at least one timestamp is wrong; cannot tell which", "5.2")
register("orders", "order_delivered_customer_date < order_delivered_carrier_date", "Delivered before carrier hand-off (impossible)",
         v["customer_date < carrier_date"], "Medium", "Flag; exclude from carrier-to-customer transit metrics",
         "Impossible sequence; customer-facing delivery time (purchase→delivery) is still valid", "5.2")
register("orders", "order_delivered_carrier_date < order_approved_at", "Carrier hand-off before approval (suspicious)",
         v["carrier_date < approved_at"], "Low", "Flag only; do not use approval→carrier interval without care",
         "Explainable by late recording of approval; mostly small offsets", "5.2")
register("orders", "order_delivered_customer_date < order_approved_at", "Delivered before approval (suspicious)",
         v["customer_date < approved_at"], "Low", "Flag only", "Same explanation as above; small subset", "5.2")

### 5.3 Status vs. timestamp consistency

`order_status` should agree with the timestamps that are present. A delivered order should have a delivery date, and a canceled order should not.

In [27]:
status = orders["order_status"]
status_rules = pd.DataFrame([
    {"rule": "status=delivered but customer_date missing",  "violations": int((status.eq("delivered") & D.isna()).sum())},
    {"rule": "status=delivered but carrier_date missing",   "violations": int((status.eq("delivered") & C.isna()).sum())},
    {"rule": "status=delivered but approved_at missing",    "violations": int((status.eq("delivered") & A.isna()).sum())},
    {"rule": "status≠delivered but customer_date present",  "violations": int((~status.eq("delivered") & D.notna()).sum())},
    {"rule": "status=shipped but carrier_date missing",     "violations": int((status.eq("shipped") & C.isna()).sum())},
    {"rule": "status=canceled with a delivery date",        "violations": int((status.eq("canceled") & D.notna()).sum())},
])
display(status_rules)
sr = status_rules.set_index("rule")["violations"]
register("orders", "order_status vs order_delivered_customer_date", "'delivered' orders without a delivery date",
         sr["status=delivered but customer_date missing"], "Medium",
         "Flag; exclude from delivery-time metrics; keep in sales metrics", "Status and timestamps disagree", "5.3")
register("orders", "order_status vs order_delivered_customer_date", "'canceled' orders that have a delivery date",
         sr["status=canceled with a delivery date"], "Medium",
         "Flag; decide business rule (canceled after delivery = return?) before revenue reporting",
         "Status and timestamps disagree; affects whether revenue is counted", "5.3")

,rule,violations
0,status=delivered but customer_date missing,8
1,status=delivered but carrier_date missing,2
2,status=delivered but approved_at missing,14
3,status≠delivered but customer_date present,6
4,status=shipped but carrier_date missing,0
5,status=canceled with a delivery date,6


### 5.4 Suspicious timestamps and time coverage

Values that are *possible* but unusual are examined here: very long delivery or approval times, and months with very few orders. The last matters for any time-series or demand analysis.

In [28]:
delivery_days = (D - P).dt.total_seconds() / 86400
approval_hours = (A - P).dt.total_seconds() / 3600
late = D.dt.normalize() > E
susp = pd.DataFrame([
    {"check": "delivery time > 60 days",            "orders": int((delivery_days > 60).sum())},
    {"check": "delivery time > 100 days",           "orders": int((delivery_days > 100).sum())},
    {"check": "approval > 7 days after purchase",   "orders": int((approval_hours > 24 * 7).sum())},
    {"check": "delivered after estimated date",     "orders": int(late.sum())},
])
display(susp)
print("delivery days:", delivery_days.describe(percentiles=[.5, .9, .99]).round(2).to_dict())

,check,orders
0,delivery time > 60 days,306
1,delivery time > 100 days,64
2,approval > 7 days after purchase,81
3,delivered after estimated date,6535


delivery days: {'count': 96476.0, 'mean': 12.56, 'std': 9.55, 'min': 0.53, '50%': 10.22, '90%': 23.1, '99%': 46.05, 'max': 209.63}


In [29]:
monthly = P.dt.to_period("M").value_counts().sort_index()
all_months = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
monthly = monthly.reindex(all_months, fill_value=0)  # reindexes a derived count Series, not the data
median_month = monthly.median()
sparse = monthly[monthly < 0.05 * median_month]
display(monthly.rename("orders").to_frame().T)
print(f"Median orders/month: {median_month:,.0f}. Months below 5% of median: {[str(m) for m in sparse.index]}")

register("orders", "order_purchase_timestamp", "Sparse/incomplete months at both ends of the period",
         int(sparse.sum()), "Medium",
         "Restrict time-series and demand analyses to complete months (approx. 2017-01 to 2018-08); document the window",
         "2016-09/11/12 and 2018-09/10 have 0–16 orders each (Nov 2016 has none); 2016-10 is also partial — would distort trends", "5.4")
register("orders", "delivery time (delivered − purchase)", "Very long delivery times (> 60 days)",
         int((delivery_days > 60).sum()), "Info",
         "Keep; analyse separately or cap only in specific models",
         "Possible (remote regions, carrier problems); no impossible values; no evidence of error", "5.4")

,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10
orders,4,324,0,1,800,1780,2682,2404,3700,3245,4026,4331,4285,4631,7544,5673,7269,6728,7211,6939,6873,6167,6292,6512,16,4


Median orders/month: 4,156. Months below 5% of median: ['2016-09', '2016-11', '2016-12', '2018-09', '2018-10']


### 5.5 Review dates

In [30]:
rev_ord = reviews[["review_id", "order_id"]].assign(RC=RC, RA=RA).merge(
    pd.DataFrame({"order_id": orders["order_id"], "P": P, "D": D}), on="order_id", how="left", validate="many_to_one")
review_rules = pd.DataFrame([
    {"rule": "answer before creation",                   "violations": int((rev_ord["RA"] < rev_ord["RC"]).sum())},
    {"rule": "review created before purchase date",      "violations": int((rev_ord["RC"] < rev_ord["P"].dt.normalize()).sum())},
    {"rule": "review created before delivery date",      "violations": int((rev_ord["RC"] < rev_ord["D"].dt.normalize()).sum())},
])
display(review_rules)
rr = review_rules.set_index("rule")["violations"]
register("reviews", "review_creation_date < purchase date", "Review created before the order was placed", rr["review created before purchase date"],
         "Low", "Flag; exclude from review-timing analyses", "Impossible; very small subset", "5.5")
register("reviews", "review_creation_date < delivery date", "Review created before delivery", rr["review created before delivery date"],
         "Info", "Keep; segment 'reviewed before delivery' when analysing satisfaction vs. delivery",
         "Plausible: survey can be triggered before delivery or customer reviews a late order — often with low scores", "5.5")

,rule,violations
0,answer before creation,0
1,review created before purchase date,64
2,review created before delivery date,5127


In [31]:
# Are early reviews associated with a different score? (context for the 'Info' classification)
early = rev_ord["RC"] < rev_ord["D"].dt.normalize()
display(reviews.assign(early_review=early.values).groupby("early_review")["review_score"].agg(["count", "mean"]).round(2))

,count,mean
early_review,,
False,94097,4.20
True,5127,1.99


## 6. Numerical validation

Each numeric column is checked against **domain rules**, meaning values that are impossible by definition (negative prices, review scores outside 1–5). Values that are allowed but unusual (zero freight, zero payment) are examined in context before being judged.

In [32]:
BRAZIL_BBOX = {"lat": (-33.75, 5.27), "lng": (-73.99, -34.79)}  # approximate national extent

domain = pd.DataFrame([
    {"dataset": "order_items", "rule": "price <= 0",                   "violations": int((items["price"] <= 0).sum())},
    {"dataset": "order_items", "rule": "freight_value < 0",            "violations": int((items["freight_value"] < 0).sum())},
    {"dataset": "order_items", "rule": "freight_value == 0",           "violations": int((items["freight_value"] == 0).sum())},
    {"dataset": "payments",    "rule": "payment_value < 0",            "violations": int((payments["payment_value"] < 0).sum())},
    {"dataset": "payments",    "rule": "payment_value == 0",           "violations": int((payments["payment_value"] == 0).sum())},
    {"dataset": "payments",    "rule": "payment_installments < 1",     "violations": int((payments["payment_installments"] < 1).sum())},
    {"dataset": "payments",    "rule": "installments > 1 on non-credit-card",
     "violations": int(((payments["payment_type"] != "credit_card") & (payments["payment_installments"] > 1)).sum())},
    {"dataset": "reviews",     "rule": "review_score not in 1..5",     "violations": int((~reviews["review_score"].isin([1, 2, 3, 4, 5])).sum())},
    {"dataset": "products",    "rule": "weight_g <= 0",                "violations": int((products["product_weight_g"] <= 0).sum())},
    {"dataset": "products",    "rule": "any dimension <= 0",
     "violations": int((products[["product_length_cm", "product_height_cm", "product_width_cm"]] <= 0).any(axis=1).sum())},
    {"dataset": "products",    "rule": "photos_qty < 1",               "violations": int((products["product_photos_qty"] < 1).sum())},
    {"dataset": "geolocation", "rule": "lat/lng outside Brazil bounding box",
     "violations": int((~geo["geolocation_lat"].between(*BRAZIL_BBOX["lat"]) | ~geo["geolocation_lng"].between(*BRAZIL_BBOX["lng"])).sum())},
])
display(domain)

,dataset,rule,violations
0,order_items,price <= 0,0
1,order_items,freight_value < 0,0
2,order_items,freight_value == 0,383
3,payments,payment_value < 0,0
4,payments,payment_value == 0,9
5,payments,payment_installments < 1,2
6,payments,installments > 1 on non-credit-card,0
7,reviews,review_score not in 1..5,0
8,products,weight_g <= 0,4
9,products,any dimension <= 0,0


### 6.1 Prices and freight

In [33]:
print("price        :", items["price"].describe().round(2).to_dict())
print("freight_value:", items["freight_value"].describe().round(2).to_dict())
fz = items[items["freight_value"] == 0]
print(f"\nzero-freight items: {len(fz):,} | distinct sellers: {fz['seller_id'].nunique()} | "
      f"share from top seller: {fz['seller_id'].value_counts(normalize=True).iloc[0]:.1%}")
fp = items["freight_value"] > items["price"]
print(f"freight > price: {fp.sum():,} items | median price of those items: {items.loc[fp, 'price'].median():.2f}")
register("order_items", "freight_value == 0", "Zero freight", len(fz), "Info",
         "Keep; treat as free shipping", "Plausible (promotions / seller-paid shipping); concentrated in few sellers", "6.1")
register("order_items", "freight_value > price", "Freight exceeds item price", int(fp.sum()), "Info",
         "Keep", "Plausible for cheap items (median price of these items is low)", "6.1")

price        : {'count': 112650.0, 'mean': 120.65, 'std': 183.63, 'min': 0.85, '25%': 39.9, '50%': 74.99, '75%': 134.9, 'max': 6735.0}
freight_value: {'count': 112650.0, 'mean': 19.99, 'std': 15.81, 'min': 0.0, '25%': 13.08, '50%': 16.26, '75%': 21.15, 'max': 409.68}

zero-freight items: 383 | distinct sellers: 9 | share from top seller: 41.3%
freight > price: 4,124 items | median price of those items: 13.90


### 6.2 Payments

In [34]:
display(payments["payment_type"].value_counts().rename("rows").to_frame().T)
zero_pay = payments[payments["payment_value"] == 0].merge(orders[["order_id", "order_status"]], on="order_id", how="left")
display(zero_pay)
inst0 = payments[payments["payment_installments"] < 1]
display(inst0)

payment_type,credit_card,boleto,voucher,debit_card,not_defined
rows,76795,19784,5775,1529,3


,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,shipped
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,delivered
3,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
4,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
5,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,delivered
6,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,shipped
7,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled
8,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,delivered


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


**Reconciliation check.** For every order, the sum of payments should equal the sum of item prices plus freight. This is the strongest available test of the monetary columns, because it compares two independently recorded tables.

In [35]:
items_total = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1).rename("items_total")
pay_total = payments.groupby("order_id")["payment_value"].sum().rename("payment_total")
recon = pd.concat([items_total, pay_total], axis=1, join="inner")
recon["diff"] = (recon["payment_total"] - recon["items_total"]).round(2)
tol = 0.01
mismatch = recon["diff"].abs() > tol
print(f"orders compared: {len(recon):,} | mismatch > {tol}: {mismatch.sum():,} ({mismatch.mean():.2%}) | > 1.00: {(recon['diff'].abs() > 1).sum():,}")
print("difference where mismatched:", recon.loc[mismatch, "diff"].describe().round(2).to_dict())
print(f"overpaid (payments > items): {(recon['diff'] > tol).sum():,} | underpaid: {(recon['diff'] < -tol).sum():,}")

register("payments", "payment_value == 0", "Zero-value payments", len(zero_pay), "Low",
         "Keep; exclude from average-ticket statistics",
         "6 zero-value voucher rows inside multi-payment orders and 3 'not_defined' rows on canceled orders", "6.2")
register("payments", "payment_type == 'not_defined'", "Undefined payment type", int((payments["payment_type"] == "not_defined").sum()), "Low",
         "Keep; map to 'unknown'", "All on canceled orders", "6.2")
register("payments", "payment_installments == 0", "Zero instalments on credit card", len(inst0), "Low",
         "Flag; treat as 1 instalment only if a business rule confirms it", "Impossible as stated (min should be 1)", "6.2")
register("payments + order_items","sum(payment_value) = sum(price + freight) per order", "Payments do not reconcile with item totals",
         int(mismatch.sum()), "Medium",
         "Use order_items for product revenue and payments for cash collected; never mix; document the gap",
         "Mostly over-payments; cause not verifiable from the data (possible instalment interest or adjustments)", "6.2")

orders compared: 98,665 | mismatch > 0.01: 303 (0.31%) | > 1.00: 249
difference where mismatched: {'count': 303.0, 'mean': 9.48, 'std': 18.08, 'min': -51.62, '25%': 1.36, '50%': 5.41, '75%': 11.83, 'max': 182.81}
overpaid (payments > items): 264 | underpaid: 39


### 6.3 Review scores

In [36]:
display(reviews["review_score"].value_counts().sort_index().rename("reviews").to_frame().T)
print("All scores in {1,2,3,4,5}:", reviews["review_score"].isin([1, 2, 3, 4, 5]).all())

review_score,1,2,3,4,5
reviews,11424,3151,8179,19142,57328


All scores in {1,2,3,4,5}: True


### 6.4 Product dimensions and weights

Beyond the domain rules, implied density (weight ÷ volume) is used as a **screening heuristic**. Most consumer goods, including their packaging, fall roughly between 0.01 and 2 g/cm³.

The threshold of **> 2 g/cm³ is a review trigger, not proof of an error**. Dense goods exist (metal parts, tools), and the recorded dimensions may describe packaging rather than the product. Products above the threshold are therefore treated as **suspicious observations to review**. They are not classified as errors and not excluded.

In [37]:
dims = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
display(products[dims].describe(percentiles=[.01, .5, .99]).round(1))
volume = products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
density = products["product_weight_g"] / volume
print("density g/cm³:", density.describe(percentiles=[.001, .01, .5, .99, .999]).round(4).to_dict())
zero_w = products[products["product_weight_g"] == 0]
display(zero_w[["product_id", "product_category_name", *dims]])
dense = products[density > 2].assign(density=density[density > 2].round(2))
print(f"products with density > 2 g/cm³ (suspicious, for review): {len(dense)}")
display(dense.sort_values("density", ascending=False).head(8)[["product_id", "product_category_name", *dims, "density"]])

register("products", "product_weight_g == 0", "Zero weight", len(zero_w), "Medium",
         "Flag; set to missing in processed layer for freight/logistics models", "A physical product cannot weigh 0 g", "6.4")
register("products", "weight / volume > 2 g/cm³", "Suspicious density observation (heuristic screen)", len(dense), "Low",
         "Flag for manual review only; do not classify as error or exclude automatically",
         "Heuristic threshold, not proof: could be a weight/dimension entry error, packaging dimensions, or a genuinely dense item", "6.4")

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.0,32949.0,32949.0,32949.0
mean,2276.5,30.8,16.9,23.2
std,4282.0,16.9,13.6,12.1
min,0.0,7.0,2.0,6.0
1%,60.0,16.0,2.0,11.0
50%,700.0,25.0,13.0,20.0
99%,22538.0,100.0,69.0,63.0
max,40425.0,105.0,105.0,118.0


density g/cm³: {'count': 32949.0, 'mean': 0.2037, 'std': 1.0093, 'min': 0.0, '0.1%': 0.0032, '1%': 0.0125, '50%': 0.1166, '99%': 1.2188, '99.9%': 7.2745, 'max': 85.2273}


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,30.0,25.0,30.0


products with density > 2 g/cm³ (suspicious, for review): 159


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,density
32491,2b752ed328ea866e4721ca4e236a416c,moveis_cozinha_area_de_servico_jantar_e_jardim,30000.0,16.0,2.0,11.0,85.23
27828,672ffb2231575afa70f7fee73fe400a1,automotivo,23200.0,16.0,2.0,11.0,65.91
17560,fec8c282124bc6f504cbc6e9ec38450a,automotivo,22100.0,16.0,2.0,11.0,62.78
16215,194dad3cf9aa121860ba80cca80331bb,automotivo,16950.0,16.0,2.0,11.0,48.15
31772,30f902329cc4fd7dba514f7a5b629b2d,pet_shop,16150.0,16.0,2.0,11.0,45.88
12924,00ffe57f0110d73fd84d162252b2c784,utilidades_domesticas,16000.0,16.0,2.0,11.0,45.45
15848,29abbd57526dab494a667e62d361f6cf,automotivo,14100.0,16.0,2.0,11.0,40.06
5540,b0c7a71c8620bd389e240f63a507dc50,livros_interesse_geral,13300.0,16.0,2.0,11.0,37.78


Many of the densest products share the **same dimensions, 16 × 2 × 11 cm**. One **possible** explanation is a packaging or shipping convention: a standard small-parcel size entered when the real dimensions were not recorded. This is a hypothesis only. It has **not been verified** against an authoritative carrier specification, and the data alone cannot confirm it.

The measurements below show how common the combination is and how much of the suspicious-density group it accounts for. Two different proportions are reported so they are not confused:

- the **share of suspicious-density products that are 16 × 2 × 11**, which is the share of the screen this pattern accounts for;
- the **share of 16 × 2 × 11 products that are suspicious-density**, which shows how often the pattern produces a flag.

In [38]:
triples = products.groupby(["product_length_cm", "product_height_cm", "product_width_cm"]).size().sort_values(ascending=False)
display(triples.head(5).rename("products").to_frame())
is_16_2_11 = (products["product_length_cm"].eq(16) & products["product_height_cm"].eq(2) & products["product_width_cm"].eq(11))
print(f"products with 16x2x11: {is_16_2_11.sum():,} | weight percentiles (g): "
      f"{products.loc[is_16_2_11, 'product_weight_g'].quantile([.5, .9, .99]).to_dict()}")
suspicious_density = density > 2                      # NaN density (missing dims/weight) evaluates to False
n_suspicious = int(suspicious_density.sum())
n_suspicious_16_2_11 = int(is_16_2_11[suspicious_density].sum())
n_16_2_11 = int(is_16_2_11.sum())
share_suspicious_that_are_16_2_11 = n_suspicious_16_2_11 / n_suspicious if n_suspicious else float("nan")
share_16_2_11_that_are_suspicious = n_suspicious_16_2_11 / n_16_2_11 if n_16_2_11 else float("nan")

print(f"suspicious-density products (density > 2)            : {n_suspicious:,}")
print(f"  ... of which have 16x2x11 dimensions                : {n_suspicious_16_2_11:,}")
print(f"share of suspicious-density products that are 16x2x11 : {share_suspicious_that_are_16_2_11:.1%}")
print(f"share of 16x2x11 products that are suspicious-density : {share_16_2_11_that_are_suspicious:.1%}")

register("products", "dimensions == 16 x 2 x 11 cm", "Most frequent dimension triple — possible packaging/default convention (unverified)",
         n_16_2_11, "Low",
         "Flag; treat these dimensions with caution in volume/density analyses; verify with the data owner",
         f"Over-represented size; accounts for {share_suspicious_that_are_16_2_11:.1%} of suspicious-density products "
         f"({n_suspicious_16_2_11} of {n_suspicious}); convention hypothesis not verified", "6.4")

,,,products
product_length_cm,product_height_cm,product_width_cm,
16.0,2.0,11.0,588
20.0,20.0,20.0,562
16.0,16.0,16.0,418
30.0,30.0,30.0,302
16.0,10.0,16.0,299


products with 16x2x11: 588 | weight percentiles (g): {0.5: 250.0, 0.9: 700.0, 0.99: 14346.99999999999}
suspicious-density products (density > 2)            : 159
  ... of which have 16x2x11 dimensions                : 58
share of suspicious-density products that are 16x2x11 : 36.5%
share of 16x2x11 products that are suspicious-density : 9.9%


### 6.5 Geolocation coordinates

In [39]:
outside = ~geo["geolocation_lat"].between(*BRAZIL_BBOX["lat"]) | ~geo["geolocation_lng"].between(*BRAZIL_BBOX["lng"])
display(geo[outside].head(10))
print(f"rows outside Brazil: {outside.sum()} | distinct zip prefixes: {geo.loc[outside, 'geolocation_zip_code_prefix'].nunique()}")
register("geolocation", "geolocation_lat / geolocation_lng", "Coordinates outside Brazil", int(outside.sum()), "Medium",
         "Exclude these points when aggregating coordinates per zip", "All addresses are Brazilian; points in other continents are errors", "6.5")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


rows outside Brazil: 42 | distinct zip prefixes: 21


## 7. Categorical validation

### 7.1 Controlled vocabularies

The columns with a small, closed set of values are checked against their expected domain. For states, the domain is the **27 Brazilian federative units**. For the others, the observed set is listed for review.

In [40]:
BR_STATES = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}
state_check = pd.DataFrame([{"dataset": a, "column": c, "distinct": datasets[a][c].nunique(),
                             "invalid_values": sorted(set(datasets[a][c].dropna()) - BR_STATES)}
                            for a, c in [("customers", "customer_state"), ("sellers", "seller_state"), ("geolocation", "geolocation_state")]])
display(state_check)
for a, c in [("orders", "order_status"), ("payments", "payment_type")]:
    print(f"{a}.{c}:", datasets[a][c].value_counts().to_dict())

,dataset,column,distinct,invalid_values
0,customers,customer_state,27,[]
1,sellers,seller_state,23,[]
2,geolocation,geolocation_state,27,[]


orders.order_status: {'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
payments.payment_type: {'credit_card': 76795, 'boleto': 19784, 'voucher': 5775, 'debit_card': 1529, 'not_defined': 3}


### 7.2 Product category names

In [41]:
cats = pd.Series(sorted(products["product_category_name"].dropna().unique()))
suffix_2 = cats[cats.str.endswith("_2")]
print("Portuguese categories with a '_2' twin:", [(c, c[:-2] in set(cats)) for c in suffix_2])

eng = translation["product_category_name_english"]
SUSPECTED_TYPOS = {"costruction": "construction", "fashio_": "fashion_", "confort": "comfort"}
typos = eng[eng.str.contains("|".join(SUSPECTED_TYPOS))]
print("English names with suspected typos:", typos.tolist())
print("Translation names inconsistent in twins:",
      eng[eng.str.contains("home_conf|home_comf")].tolist())

untrans = products["product_category_name"].notna() & ~products["product_category_name"].isin(translation["product_category_name"])
print("Untranslated categories:", products.loc[untrans, "product_category_name"].value_counts().to_dict())

register("products", "product_category_name", "Near-duplicate categories ('_2' twins)", int(products["product_category_name"].isin(suffix_2).sum()),
         "Low", "Decide whether to merge twins in a category mapping table (business decision)",
         "casa_conforto/_2 and eletrodomesticos/_2 look like the same category split in two", "7.2")
register("translation", "product_category_name_english", "Typos in English names", len(typos), "Low",
         "Correct in a project-owned mapping table (never in the raw file)", "'costruction', 'fashio_', 'confort' — reporting labels only", "7.2")
register("products", "product_category_name", "Category has no English translation", int(untrans.sum()), "Low",
         "Add the 2 missing translations to the project mapping table", "pc_gamer and portateis_cozinha_e_preparadores_de_alimentos", "7.2")

Portuguese categories with a '_2' twin: [('casa_conforto_2', True), ('eletrodomesticos_2', True)]
English names with suspected typos: ['costruction_tools_garden', 'home_confort', 'costruction_tools_tools', 'fashio_female_clothing']
Translation names inconsistent in twins: ['home_confort', 'home_comfort_2']
Untranslated categories: {'portateis_cozinha_e_preparadores_de_alimentos': 10, 'pc_gamer': 3}


### 7.3 City names

City names are free text. **Decision:** to measure *inconsistency* without changing the data, a temporary normalised key is computed (accents removed, lower case, whitespace trimmed). If two raw spellings share a key, they are variants of the same city. Other patterns checked: digits, URL encoding (`%26`), embedded state suffixes (`city / sp`), and e-mail addresses.

In [42]:
def norm_key(s: pd.Series) -> pd.Series:
    return s.map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode().lower().strip()
                 if isinstance(x, str) else x)

city_rows = []
PATTERNS = {"has_digit": r"\d", "url_encoded": r"%[0-9a-fA-F]{2}", "state_suffix": r"[/\\,-]\s*[a-z]{2}\s*$|/",
            "email": r"@", "non_ascii": r"[^\x00-\x7f]"}
for a, c in [("customers", "customer_city"), ("sellers", "seller_city"), ("geolocation", "geolocation_city")]:
    s = datasets[a][c]
    row = {"dataset": a, "column": c, "distinct_raw": s.nunique(), "distinct_normalised": norm_key(s).nunique()}
    row["spelling_variants"] = row["distinct_raw"] - row["distinct_normalised"]
    for k, pat in PATTERNS.items():
        row[f"rows_{k}"] = int(s.str.contains(pat, regex=True).sum())
    city_rows.append(row)
city_df = pd.DataFrame(city_rows)
display(city_df)
print("Seller city values with digits / separators / e-mail:")
print(sellers.loc[sellers["seller_city"].str.contains(r"[\d/\\@,]|\s-\s", regex=True), "seller_city"].unique().tolist())
print("\nGeolocation city values with URL encoding:", geo.loc[geo["geolocation_city"].str.contains("%", regex=False), "geolocation_city"].unique().tolist())

register("geolocation", "geolocation_city", "Spelling variants (accents), URL-encoded values",
         int(city_df.query("dataset=='geolocation'")["rows_non_ascii"].iloc[0]), "Low",
         "Use a normalised key (accent-free, lower case) for matching; decode '%26apos%3b' → apostrophe",
         "Same city recorded several ways; affects grouping by city", "7.3")
register("sellers", "seller_city", "Malformed city values (state suffixes, digits, e-mail)",
         int(sellers["seller_city"].str.contains(r"[\d/\\@,]|\s-\s", regex=True).sum()), "Low",
         "Standardise via normalised key + manual mapping; prefer zip prefix/state for geography",
         "Free-text entry errors; small number of sellers", "7.3")

,dataset,column,distinct_raw,distinct_normalised,spelling_variants,rows_has_digit,rows_url_encoded,rows_state_suffix,rows_email,rows_non_ascii
0,customers,customer_city,4119,4119,0,1,0,8,0,0
1,sellers,seller_city,611,609,2,1,0,20,1,3
2,geolocation,geolocation_city,8011,5966,2045,11,2,98,0,73442


Seller city values with digits / separators / e-mail:
['lages - sc', 'auriflama/sp', 'sao paulo / sao paulo', '04482255', 'novo hamburgo, rio grande do sul, brasil', 'cariacica / es', 'sao paulo - sp', 'sbc/sp', 'santo andre/sao paulo', 'sp / sp', 'maua/sao paulo', 'mogi das cruzes / sp', 'rio de janeiro \\rio de janeiro', 'barbacena/ minas gerais', 'rio de janeiro / rio de janeiro', 'pinhais/pr', 'ribeirao preto / sao paulo', 'carapicuiba / sao paulo', 'vendas@creditparts.com.br', 'sao sebastiao da grama/sp', 'rio de janeiro, rio de janeiro, brasil', 'jacarei / sao paulo']

Geolocation city values with URL encoding: ['são joão do pau d%26apos%3balho', 'lambari d%26apos%3boeste']


## 8. Outlier investigation

### 8.1 Method

**Decision: IQR (Tukey) fences rather than z-scores.** Z-scores assume an approximately normal distribution, and these monetary and physical columns are strongly right-skewed (skewness is measured below). IQR fences use quartiles and are robust to the tail itself.

- **Mild outliers:** outside [Q1 − 1.5·IQR, Q3 + 1.5·IQR]
- **Extreme outliers:** outside [Q1 − 3·IQR, Q3 + 3·IQR]

For skewed, positive data the raw-scale IQR flags much of the **natural right tail**. The fences are therefore also computed on the **log scale** (`log1p`), which tests whether a value is unusual *relative to a log-normal shape*. Comparing the two counts shows how many "outliers" are just the shape of the distribution.

**Flagged ≠ wrong.** Every flag is checked further in 8.3.

In [43]:
def iqr_profile(name: str, s: pd.Series) -> dict:
    s = s.dropna().astype(float)
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    lq1, lq3 = np.log1p(s).quantile([.25, .75])
    liqr = lq3 - lq1
    log_out = ((np.log1p(s) < lq1 - 1.5 * liqr) | (np.log1p(s) > lq3 + 1.5 * liqr)).sum()
    return {"variable": name, "n": len(s), "min": s.min(), "q1": q1, "median": s.median(), "q3": q3, "max": s.max(),
            "skew": round(stats.skew(s), 2), "kurtosis": round(stats.kurtosis(s), 1),
            "upper_fence": round(hi, 2), "mild_outliers": int(((s < lo) | (s > hi)).sum()),
            "extreme_outliers": int(((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).sum()),
            "mild_%": round(((s < lo) | (s > hi)).mean() * 100, 2), "log_scale_outliers": int(log_out)}

OUTLIER_VARS = {
    "order_items.price": items["price"], "order_items.freight_value": items["freight_value"],
    "payments.payment_value": payments["payment_value"], "payments.payment_installments": payments["payment_installments"],
    "products.weight_g": products["product_weight_g"], "products.length_cm": products["product_length_cm"],
    "products.height_cm": products["product_height_cm"], "products.width_cm": products["product_width_cm"],
    "products.photos_qty": products["product_photos_qty"], "products.description_len": products["product_description_lenght"],
    "orders.delivery_days": delivery_days, "orders.approval_hours": approval_hours,
}
outlier_df = pd.DataFrame([iqr_profile(k, v) for k, v in OUTLIER_VARS.items()]).set_index("variable")
display(outlier_df.round(2))

,n,min,q1,median,q3,max,skew,kurtosis,upper_fence,mild_outliers,extreme_outliers,mild_%,log_scale_outliers
variable,,,,,,,,,,,,,
order_items.price,112650,0.85,39.90,74.99,134.90,6735.00,7.92,120.8,277.40,8427,4074,7.48,1471
order_items.freight_value,112650,0.00,13.08,16.26,21.15,409.68,5.64,59.8,33.25,12134,5538,10.77,7961
payments.payment_value,103886,0.00,56.79,100.00,171.84,13664.08,9.25,241.8,344.41,7981,4046,7.68,2931
payments.payment_installments,103886,0.00,1.00,1.00,4.00,24.00,1.66,2.5,8.50,6313,169,6.08,40
products.weight_g,32949,0.00,300.00,700.00,1900.00,40425.00,3.60,15.1,4300.00,4551,3073,13.81,10
products.length_cm,32949,7.00,18.00,25.00,38.00,105.00,1.75,3.5,68.00,1380,349,4.19,0
products.height_cm,32949,2.00,8.00,13.00,21.00,105.00,2.14,6.7,40.50,1892,559,5.74,130
products.width_cm,32949,6.00,15.00,20.00,30.00,118.00,1.67,4.1,52.50,912,108,2.77,72
products.photos_qty,32341,1.00,1.00,1.00,3.00,20.00,2.19,7.3,6.00,849,209,2.63,114


### 8.2 Distribution shape

Quantile summaries for the main monetary variables show where the mass of the data sits and how long the tail is. They are printed as numbers (no charts, since this is not the EDA phase).

In [44]:
qs = [.01, .05, .25, .5, .75, .95, .99, .999]
display(pd.DataFrame({k: OUTLIER_VARS[k].quantile(qs) for k in
                      ["order_items.price", "order_items.freight_value", "payments.payment_value", "orders.delivery_days"]}).round(2))

,order_items.price,order_items.freight_value,payments.payment_value,orders.delivery_days
0.010,9.99,4.42,6.69,1.83
0.050,17.00,7.78,26.11,3.02
0.250,39.90,13.08,56.79,6.77
0.500,74.99,16.26,100.00,10.22
0.750,134.90,21.15,171.84,15.72
0.950,349.90,45.12,437.64,29.28
0.990,890.00,84.52,1039.92,46.05
0.999,2110.00,175.59,2339.23,83.14


### 8.3 Genuine extremes or errors?

Each high-value flag is checked against **independent evidence**:

1. **Expensive items:** do they belong to categories where high prices are expected?
2. **Large payments:** does the payment agree with the order's own item total (from a different table)? If so, the large value is corroborated.
3. **High freight:** is it explained by product weight? This is measured with a Spearman rank correlation (robust to skew).
4. **Heavy products:** does the category make sense?

In [45]:
cat_en = products[["product_id", "product_category_name"]].merge(translation, on="product_category_name", how="left")
top_price = (items.nlargest(10, "price")[["order_id", "product_id", "price", "freight_value"]]
             .merge(cat_en[["product_id", "product_category_name_english"]], on="product_id", how="left"))
display(top_price)

,order_id,product_id,price,freight_value,product_category_name_english
0,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31,housewares
1,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21,computers
2,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66,art
3,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34,small_appliances
4,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34,small_appliances
5,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78,computers
6,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45,musical_instruments
7,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27,consoles_games
8,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51,sports_leisure
9,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01,watches_gifts


In [46]:
big_pay = recon[recon["payment_total"] > recon["payment_total"].quantile(.99)]
corroborated = (big_pay["diff"].abs() <= 1).mean()
print(f"orders in top 1% by payment: {len(big_pay):,} | payment matches item total (±1): {corroborated:.1%}")

orders in top 1% by payment: 987 | payment matches item total (±1): 99.9%


In [47]:
iw = items[["product_id", "freight_value"]].merge(products[["product_id", "product_weight_g"]], on="product_id", how="left").dropna()
rho, pval = stats.spearmanr(iw["freight_value"], iw["product_weight_g"])
print(f"Spearman(freight, weight) = {rho:.3f} (p={pval:.1e}) over {len(iw):,} items")
top_freight = items.nlargest(5, "freight_value")[["product_id", "price", "freight_value"]].merge(
    products[["product_id", "product_weight_g"]], on="product_id")
display(top_freight)
heavy = products.nlargest(5, "product_weight_g")[["product_id", "product_category_name", *dims]]
display(heavy)

Spearman(freight, weight) = 0.447 (p=0.0e+00) over 112,632 items


,product_id,price,freight_value,product_weight_g
0,ec31d2a17b299511e7c8627be9337b9b,979.00,409.68,14675.0
1,a3cd9517ebf5a50dca25acce54f3b171,2338.08,375.28,30000.0
2,a3cd9517ebf5a50dca25acce54f3b171,2338.08,375.28,30000.0
3,256a9c364b75753b97bee410c9491ad8,1149.00,339.59,25250.0
4,97c948ebc8c04b26b7bbb095d4228f2a,1050.00,338.30,30000.0


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,40425.0,13.0,65.0,28.0
344,d0877f0094337c414d23f5a3c7bad20c,moveis_escritorio,30000.0,50.0,50.0,30.0
509,53f92b0474f91fcb5bd188c6a8075c38,utilidades_domesticas,30000.0,76.0,51.0,51.0
955,ceeba7d5636e59173cc5f484e913db3d,NaN,30000.0,65.0,65.0,65.0
1159,f97ad9066c718a6cef93dfcf253d3e0d,moveis_decoracao,30000.0,61.0,70.0,65.0


In [48]:
od_ = outlier_df
register("order_items", "price (IQR)", "Statistical outliers in price (high tail)", od_.loc["order_items.price", "mild_outliers"], "Info",
         "Keep; use median/robust statistics or log scale; never remove",
         "Top prices are in categories where high prices are expected (computers, appliances, instruments, consoles) — no impossible values", "8")
register("payments", "payment_value (IQR)", "Statistical outliers in payment value", od_.loc["payments.payment_value", "mild_outliers"], "Info",
         "Keep; use robust statistics", "Large payments reconcile with their orders' item totals — corroborated by an independent table", "8")
register("order_items", "freight_value (IQR)", "Statistical outliers in freight", od_.loc["order_items.freight_value", "mild_outliers"], "Info",
         "Keep", "Freight correlates with product weight; extreme freight belongs to heavy products", "8")

## 9. Referential integrity (revisited from notebook 02)

The exceptions found in notebook 02 are recomputed here, so that this notebook is self-contained, and their **quality implications** are recorded.

In [49]:
ri = pd.DataFrame([
    {"exception": "orders without order_items", "records": int((~orders["order_id"].isin(items["order_id"])).sum())},
    {"exception": "orders without payments",    "records": int((~orders["order_id"].isin(payments["order_id"])).sum())},
    {"exception": "orders without reviews",     "records": int((~orders["order_id"].isin(reviews["order_id"])).sum())},
    {"exception": "orders with >1 review",      "records": int((reviews["order_id"].value_counts() > 1).sum())},
    {"exception": "products without category",  "records": int(products["product_category_name"].isna().sum())},
    {"exception": "products with untranslated category", "records": int(untrans.sum())},
    {"exception": "customer zip not in geolocation", "records": int((~customers["customer_zip_code_prefix"].isin(geo["geolocation_zip_code_prefix"])).sum())},
    {"exception": "seller zip not in geolocation",   "records": int((~sellers["seller_zip_code_prefix"].isin(geo["geolocation_zip_code_prefix"])).sum())},
    {"exception": "child rows with orphan keys (all core FKs)", "records": int(
        (~items["order_id"].isin(orders["order_id"])).sum() + (~payments["order_id"].isin(orders["order_id"])).sum()
        + (~reviews["order_id"].isin(orders["order_id"])).sum() + (~orders["customer_id"].isin(customers["customer_id"])).sum()
        + (~items["product_id"].isin(products["product_id"])).sum() + (~items["seller_id"].isin(sellers["seller_id"])).sum())},
]).set_index("exception")
display(ri)
no_items_status = orders.loc[~orders["order_id"].isin(items["order_id"]), "order_status"].value_counts()
display(no_items_status.rename("orders without items").to_frame().T)

register("orders", "order_id ∉ order_items", "Orders without any item", ri.loc["orders without order_items", "records"], "Medium",
         "Keep in order-status analyses; exclude from product/revenue analyses (they have no items)",
         "Mostly 'unavailable' / 'canceled' — orders that never materialised; inner joins drop them silently", "9")
register("orders", "order_id ∉ payments", "Order without payment", ri.loc["orders without payments", "records"], "Low",
         "Flag; exclude from payment analyses", "Single delivered order with no payment record", "9")
register("orders", "order_id ∉ reviews", "Orders without review", ri.loc["orders without reviews", "records"], "Info",
         "Keep; satisfaction metrics apply to reviewed orders only", "Reviews are optional", "9")
register("orders", "order_id with >1 review", "Orders with several reviews", ri.loc["orders with >1 review", "records"], "Medium",
         "Define a rule (latest review or mean score) before joining to orders", "One-to-many join inflates rows and averages", "9")
register("customers / sellers", "zip prefix ∉ geolocation", "Location without coordinates",
         ri.loc["customer zip not in geolocation", "records"] + ri.loc["seller zip not in geolocation", "records"], "Low",
         "Fall back to state-level geography for these records", "~0.3% of customers/sellers", "9")

,records
exception,
orders without order_items,775
orders without payments,1
orders without reviews,768
orders with >1 review,547
products without category,610
products with untranslated category,13
customer zip not in geolocation,278
seller zip not in geolocation,7
child rows with orphan keys (all core FKs),0


order_status,unavailable,canceled,created,invoiced,shipped
orders without items,603,164,5,2,1


**Implications.**

- The core keys have **zero orphan rows**, so integrity problems will not *lose* data when joining children to parents.
- The risks run the other way, on the **parent side**: orders without items, payments or reviews disappear in inner joins, and the one-to-many tables (reviews, payments) inflate row counts.
- Notebook 02 showed the second risk in detail: a naive items⋈payments join inflates payment value by 26.9%.
- Both risks must be managed by choosing the analysis grain and aggregating to it before joining (see notebook 02, Section 8.7).

In [50]:
# Rows a naive items ⋈ payments join would add, computed from key counts (nothing is materialised)
ipo, ppo = items["order_id"].value_counts(), payments["order_id"].value_counts()
common = ipo.index.intersection(ppo.index)
extra_rows = int((ipo[common] * ppo[common]).sum() - ipo[common].sum())
print(f"naive items⋈payments would add {extra_rows:,} duplicated rows")
register("order_items + payments", "join on order_id", "Many-to-many join duplicates rows (nb 02: +26.86% payment value)",
         extra_rows, "High",
         "Aggregate payments to order level before any join with items; use merge(validate=...) in pipelines",
         "Most damaging integration error available in this schema", "9")

naive items⋈payments would add 4,954 duplicated rows


## 10. Data quality rule register

The complete list of issues, sorted by severity and then by number of affected records. `pct_of_dataset` relates the count to the size of the dataset in the row. It is left empty for rules that span several datasets, because there is no single denominator.

In [51]:
rules_df = pd.DataFrame(RULES)
size = {a: len(d) for a, d in datasets.items()}
def pct(row):
    n = size.get(row["dataset"])  # only for single-dataset rules; multi-dataset rules have no single denominator
    return round(row["affected_records"] / n * 100, 3) if n else None
rules_df["pct_of_dataset"] = rules_df.apply(pct, axis=1)
rules_df = (rules_df.assign(_s=rules_df["severity"].map(SEVERITY_ORDER))
            .sort_values(["_s", "affected_records"], ascending=[True, False]).drop(columns="_s").reset_index(drop=True))
rules_df.index = rules_df.index + 1
rules_df.index.name = "rule_no"

with pd.option_context("display.max_colwidth", 120):
    display(rules_df[["dataset", "column_or_rule", "quality_issue", "affected_records", "pct_of_dataset",
                      "severity", "recommended_treatment", "justification"]])

,dataset,column_or_rule,quality_issue,affected_records,pct_of_dataset,severity,recommended_treatment,justification
rule_no,,,,,,,,
1,orders / order_items / reviews,8 date/time columns,Timestamps stored as text,311315,NaN,High,Parse with explicit format '%Y-%m-%d %H:%M:%S' (errors='coerce') at load time,"All values match one format and parse cleanly, but as text they cannot be compared, sorted or differenced"
2,customers / sellers / geolocation,*_zip_code_prefix,Zip prefix inferred as int64 — leading zeros lost,270755,NaN,High,"Read zip columns with dtype=str at load time (on every table, so joins stay type-consistent)","Codes like '01037' become 1037: wrong as labels, and mismatched if any other source keeps them as text"
3,customers,customer_unique_id,One person has several customer_id (one per order),6342,6.378,High,Use customer_unique_id as the customer key for all customer-level metrics,customer_id is order-level (nb 02); using it makes every customer look like a one-time buyer
4,order_items + payments,join on order_id,Many-to-many join duplicates rows (nb 02: +26.86% payment value),4954,NaN,High,Aggregate payments to order level before any join with items; use merge(validate=...) in pipelines,Most damaging integration error available in this schema
5,geolocation,all columns,Exact duplicate rows,261831,26.179,Medium,Do not use row-level; aggregate to one row per zip prefix (e.g. median lat/lng) when building a lookup,26% repeated points add no information but would multiply joins
6,reviews,review_id,review_id not unique (one review linked to several orders),1603,1.616,Medium,"Key reviews on (review_id, order_id); de-duplicate by review_id only for review-level counts",Content identical across repeats (nb 02) — counting rows would over-count reviews
7,orders,order_id ∉ order_items,Orders without any item,775,0.779,Medium,Keep in order-status analyses; exclude from product/revenue analyses (they have no items),Mostly 'unavailable' / 'canceled' — orders that never materialised; inner joins drop them silently
8,products,"category, name/description length, photos",Product metadata missing (same rows),610,1.851,Medium,Keep rows; label category as an explicit 'unknown' group in reporting,4 columns missing together = one cause (unpopulated listing); products are still sold and must not be dropped
9,orders,order_id with >1 review,Orders with several reviews,547,0.550,Medium,Define a rule (latest review or mean score) before joining to orders,One-to-many join inflates rows and averages


In [52]:
display(rules_df.groupby("severity").size().reindex(list(SEVERITY_ORDER)).rename("rules").to_frame().T)
display(rules_df.assign(dataset=rules_df["dataset"].str.split(r" [/+] ", regex=True)).explode("dataset")
        .groupby("dataset").size().rename("rules touching dataset").sort_values(ascending=False).to_frame().T)

severity,High,Medium,Low,Info
rules,4,13,22,13


dataset,orders,products,order_items,reviews,payments,geolocation,customers,sellers,translation
rules touching dataset,16,10,9,6,6,5,3,3,1


## 11. Verification

In [53]:
sha_after = {f: sha256(RAW_DIR / f) for f in FILES.values()}
memory_hash_after = {a: frame_hash(d) for a, d in datasets.items()}
processed_after = sorted(p.name for p in PROCESSED_DIR.rglob("*")) if PROCESSED_DIR.exists() else []

datasets_in_rules = set(rules_df["dataset"].str.split(r" [/+] ", regex=True).explode())
assessed_by_section = {
    "missing values (2)": set(missing_df["dataset"]),
    "exact duplicates (3.1)": set(exact_dup.index),
    "data types (4)": set(types_df["dataset"]),
}
display(pd.DataFrame({sec: {a: a in s for a in FILES} for sec, s in assessed_by_section.items()}))

checks = {
    "All 9 datasets loaded":                                 len(datasets) == 9 and not load_errors,
    "All 9 assessed for missing values":                     assessed_by_section["missing values (2)"] == set(FILES),
    "All 9 assessed for exact duplicates":                   assessed_by_section["exact duplicates (3.1)"] == set(FILES),
    "All 9 assessed for data types":                         assessed_by_section["data types (4)"] == set(FILES),
    "Rule register covers all 9 datasets":                   set(FILES) <= datasets_in_rules,
    "Raw files unchanged (SHA-256)":                         sha_before == sha_after,
    "In-memory DataFrames unchanged (content hash)":         memory_hash_before == memory_hash_after,
    "No processed files created":                            processed_after == processed_before,
}
for k, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {k}")
assert all(checks.values()), "Verification failed"
print(f"\nRules registered: {len(rules_df)}. Notebook completed — all checks passed.")

,missing values (2),exact duplicates (3.1),data types (4)
customers,True,True,True
geolocation,True,True,True
orders,True,True,True
order_items,True,True,True
payments,True,True,True
reviews,True,True,True
products,True,True,True
sellers,True,True,True
translation,True,True,True


[PASS] All 9 datasets loaded
[PASS] All 9 assessed for missing values
[PASS] All 9 assessed for exact duplicates
[PASS] All 9 assessed for data types
[PASS] Rule register covers all 9 datasets
[PASS] Raw files unchanged (SHA-256)
[PASS] In-memory DataFrames unchanged (content hash)
[PASS] No processed files created

Rules registered: 52. Notebook completed — all checks passed.


In [54]:
display(pd.DataFrame({"sha256_before": sha_before, "sha256_after": sha_after}).assign(
    unchanged=lambda d: d["sha256_before"] == d["sha256_after"]))

,sha256_before,sha256_after,unchanged
olist_customers_dataset.csv,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2,True
olist_geolocation_dataset.csv,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6,b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6,True
olist_orders_dataset.csv,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf,True
olist_order_items_dataset.csv,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279,True
olist_order_payments_dataset.csv,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0,4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0,True
olist_order_reviews_dataset.csv,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7,012b61c7593e34f51fa614efdf802b9c7056ce6aae5307ddb93236e7cfc797d7,True
olist_products_dataset.csv,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc,True
olist_sellers_dataset.csv,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e,True
product_category_name_translation.csv,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278,a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278,True


## 12. Summary of findings

### 12.1 Overall verdict

- The Olist data is **structurally sound**. There are no orphan keys, no composite-key violations, no unparseable dates, no negative monetary values and no invalid review scores or states.
- Most issues are about **representation** (types, grain, identifiers) rather than corrupted values. Genuinely invalid values affect **small numbers of records** (from a handful up to about 170 per rule).
- The register holds **52 rules**: 4 High, 13 Medium, 22 Low and 13 Info. The Info rules are documented properties, not errors.

### 12.2 By dimension

| Dimension | Key findings |
|---|---|
| **Missing values** | Only 3 of 9 tables have gaps. Order timestamp gaps **follow `order_status`** (events that have not happened yet), so they are mostly legitimate. Review comments are optional (59% / 88% missing), and missingness rises with the score. 610 products lack all descriptive metadata together (one cause). |
| **Duplicates** | Exact duplicates exist only in `geolocation` (261,831 rows, 26%). `review_id` repeats (1,603 rows). One person has several `customer_id`s (6,342 rows). Business-level repeats in `order_items` (10,225) are how **quantity** is recorded. There are 86 apparent duplicate review submissions and 582 orders by the same person in the same second (59% of groups span several sellers). No composite-key violations. |
| **Data types** | All 8 date columns are text, but all follow one format and **all parse**. All zip prefixes lost their leading zero (270,755 rows); the raw file stores 5-digit text. Three count columns are float only because of NaN. No numeric values are stored as text. |
| **Dates** | Impossible sequences: carrier hand-off before purchase (166), delivery before carrier hand-off (23), review before purchase (64). Status conflicts: 8 delivered orders without a delivery date, 6 canceled orders with one. 4 shipping deadlines fall in 2020. **Coverage gaps:** Nov 2016 has no orders, and Sept–Dec 2016 and Sept–Oct 2018 are near-empty. |
| **Numbers** | No negative prices, freight or payments. Edge cases: 383 zero-freight items, 9 zero payments, 2 credit-card payments with 0 instalments, 4 zero-weight products. 303 orders' payments do not reconcile with item totals (mostly over-payments, cause unknown). |
| **Categories** | States are all valid. There are two `_2` twin categories, 4 typos in English names and 2 untranslated categories. City names are inconsistent: 2,045 accent variants in geolocation, URL-encoded names, and seller cities containing states, digits or an e-mail address. |
| **Outliers** | All monetary and physical variables are strongly right-skewed (skew 1.7–9.3). Raw-scale IQR flags 7–14% of values. The log-scale IQR flags far fewer, so most flags are just the natural tail. Extreme prices fall in expensive categories, large payments reconcile with their orders (99.9%), and freight tracks weight (ρ = 0.45). **Conclusion: the extremes are genuine and must not be removed.** The real errors are domain violations: 42 coordinates outside Brazil and 4 products weighing 0 g. Separately, a **heuristic density screen** (> 2 g/cm³) flags **159 suspicious products for review**. These are not confirmed errors. 58 of them (36.5%) share the dimensions 16 × 2 × 11 cm, a possible packaging convention that has not been verified (only 9.9% of all 16 × 2 × 11 products are flagged). |
| **Referential integrity** | Zero orphan rows. The exceptions are on the parent side: 775 orders without items, 1 without payment, 768 without review, 547 with several reviews, 610 + 13 products without a usable category, and 285 customers/sellers without coordinates. |

### 12.3 Warnings for the cleaning and integration phase

> ⚠️ **W1 — Load-time typing (High).** Parse the 8 date columns with the explicit format, and read **all zip prefixes as text** in every table. These are load-time decisions, and fixing them later is error-prone.

> ⚠️ **W2 — Customer identity (High).** `customer_unique_id` is the customer. `customer_id` is a per-order surrogate. Any customer metric built on `customer_id` is wrong.

> ⚠️ **W3 — Grain and double counting (High).** Never join `order_items` to `payments` directly: it adds 4,954 duplicated rows and +26.86% payment value. Aggregate one-to-many tables (payments, reviews) to order level first.

> ⚠️ **W4 — Revenue definition (Medium).** Item totals and payment totals differ in 303 orders. Choose one source per metric (item-based GMV vs. cash collected) and never mix them.

> ⚠️ **W5 — Time window (Medium).** Use complete months only (about Jan 2017 – Aug 2018) for trends, seasonality and demand forecasting.

> ⚠️ **W6 — Delivery metrics (Medium).** Compute them only on delivered orders that have the required timestamps, and exclude the flagged impossible sequences. Status/timestamp conflicts need a business rule, for example whether a canceled order with a delivery date is a return.

> ⚠️ **W7 — Geolocation (Medium).** Reduce it to one row per zip prefix (robust centre), excluding the 42 out-of-country points, before any join. Fall back to state for the 285 records with no coordinates.

> ⚠️ **W8 — Do not remove outliers.** The high tail is real business (expensive products, large orders, heavy freight). Use medians, log scales or robust models instead. Only rule-based **invalid** values should be flagged.

> ⚠️ **W9 — Flag, don't delete.** Every invalid-value treatment in the register is a *flag*, so that analyses can include or exclude the records explicitly and reproducibly. Corrections to labels (translations, typos, `_2` twins, city spellings) belong in **project-owned mapping tables**, never in the raw files.

**Scope note:** this notebook only assessed the data. Nothing was cleaned, deleted, filled, converted and saved, or written to `data/processed/`. Section 11 verifies this.